# Proposed Model — CatBoost v1

We use CatBoostRegressor to capture nonlinear patterns in sales. Inputs are
15 shared calendar features and three categorical features: country, store,
and product.

We use the shared preprocessing and remove rows with missing num_sold.
The model uses the original target, fixed parameters, and seed 42.
Negative predictions are clipped to zero. No tuning or early stopping is used.

We evaluate with the shared MAPE (%) on expanding-window folds for
2014, 2015, and 2016, plus a 2014–2016 holdout trained on 2010–2013.
Results are compared with seasonal_naive_last_year from results/metrics.csv.

## Reproducibility

Place train.csv and test.csv in data/ and keep the baseline metrics in
results/metrics.csv. Install dependencies in your virtual environment:

```bash
python -m pip install -r requirements.txt
python -m pip install ipykernel nbclient
```

Run all cells in order using this environment. A fresh-kernel run reproduced
all four scores without adding duplicate CSV rows.

Tested: Python 3.14.4, NumPy 2.5.3, pandas 3.0.5, CatBoost 1.2.10.
Versions are not pinned.

## 1. Imports

In [1]:
import sys
from pathlib import Path

current_dir = Path.cwd()
candidates = [
    current_dir,
    current_dir.parent,
    current_dir / "sticker-sales-forecasting",
]

PROJECT_ROOT = next(
    (
        path
        for path in candidates
        if (path / "src" / "preprocessing.py").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError(f"Project root not found from: {current_dir}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import catboost

from src.preprocessing import load_data, preprocess_data, drop_missing_target
from src.features import add_date_features
from src.validation import expanding_window_splits, temporal_train_val_split
from src.metrics import mape
from src.model import (
    MODEL_NAME,
    MODEL_VERSION,
    FEATURE_COLS,
    CATEGORICAL_FEATURES,
    DEFAULT_PARAMS,
    fit_model,
    predict_model,
)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("CatBoost:", catboost.__version__)

Python: 3.14.4
NumPy: 2.5.3
pandas: 3.0.5
CatBoost: 1.2.10


## 2. Data Preparation

In [2]:
train_raw, test_raw = load_data(
    PROJECT_ROOT / "data" / "train.csv",
    PROJECT_ROOT / "data" / "test.csv",
)

train = preprocess_data(train_raw)
missing_target_count = int(train["num_sold"].isna().sum())

train = drop_missing_target(train)
train = add_date_features(train)

assert train[FEATURE_COLS].notna().all().all()
assert train["num_sold"].notna().all()
assert (train["num_sold"] > 0).all()
assert all(
    train[column].map(lambda value: isinstance(value, str)).all()
    for column in CATEGORICAL_FEATURES
)

print("Rows with missing target removed:", missing_target_count)
print("Prepared feature matrix:", train[FEATURE_COLS].shape)

Rows with missing target removed: 8871
Prepared feature matrix: (221259, 18)


## 3. Temporal Validation

In [3]:
folds = list(expanding_window_splits(train))
long_train, long_val = temporal_train_val_split(
    train,
    val_start_date="2014-01-01",
)

split_summary = []

splits_to_check = [
    (f"fold_{index}_{val_part['date'].dt.year.iloc[0]}", train_part, val_part)
    for index, (train_part, val_part) in enumerate(folds, start=1)
]
splits_to_check.append(("long_horizon", long_train, long_val))

for name, train_part, val_part in splits_to_check:
    assert train_part["date"].max() < val_part["date"].min(), name
    assert set(train_part["id"]).isdisjoint(val_part["id"]), name
    assert train_part["num_sold"].notna().all(), name
    assert val_part["num_sold"].notna().all(), name

    split_summary.append({
        "split": name,
        "train_start": train_part["date"].min().date(),
        "train_end": train_part["date"].max().date(),
        "val_start": val_part["date"].min().date(),
        "val_end": val_part["date"].max().date(),
        "train_rows": len(train_part),
        "val_rows": len(val_part),
    })

print(pd.DataFrame(split_summary).to_string(index=False))
print("Temporal split checks passed.")

       split train_start  train_end  val_start    val_end  train_rows  val_rows
 fold_1_2014  2010-01-01 2013-12-31 2014-01-01 2014-12-31      126026     31823
 fold_2_2015  2010-01-01 2014-12-31 2015-01-01 2015-12-31      157849     31643
 fold_3_2016  2010-01-01 2015-12-31 2016-01-01 2016-12-31      189492     31767
long_horizon  2010-01-01 2013-12-31 2014-01-01 2016-12-31      126026     95233
Temporal split checks passed.


## 4. Expanding-Window Evaluation

In [4]:
train_part, val_part = folds[0]

print("Training fold_1_2014")

model_2014 = fit_model(train_part)
predictions_2014 = predict_model(model_2014, val_part)

print("Training completed.")
print("Prediction min:", predictions_2014.min())
print("Prediction max:", predictions_2014.max())
print("Prediction mean:", predictions_2014.mean())
print("Negative predictions:", int((predictions_2014 < 0).sum()))
print("All predictions finite:", bool(np.isfinite(predictions_2014).all()))

mape_2014 = mape(val_part["num_sold"].to_numpy(), predictions_2014)
print(f"MAPE 2014: {mape_2014:.6f}%")

Training fold_1_2014


Clipping 716 negative predictions to zero (raw minimum: -19.682133).
Training completed.
Prediction min: 0.0
Prediction max: 4925.9072151098235
Prediction mean: 827.70091953269
Negative predictions: 0
All predictions finite: True
MAPE 2014: 15.917685%


In [5]:
raw_predictions_2014 = model_2014.predict(
    folds[0][1][FEATURE_COLS]
)

raw_mape_2014 = mape(
    folds[0][1]["num_sold"].to_numpy(),
    raw_predictions_2014,
)

print("Raw negative predictions:", int((raw_predictions_2014 < 0).sum()))
print(f"MAPE before clipping: {raw_mape_2014:.6f}%")
print(f"MAPE after clipping: {mape_2014:.6f}%")

Raw negative predictions: 716
MAPE before clipping: 18.050226%
MAPE after clipping: 15.917685%


In [6]:
fold_scores = [
    {
        "fold": "fold_1_2014",
        "mape": mape_2014,
    }
]

for fold_number, (train_part, val_part) in enumerate(folds[1:], start=2):
    year = int(val_part["date"].dt.year.iloc[0])
    fold_name = f"fold_{fold_number}_{year}"

    print(f"\nTraining {fold_name}")

    fold_model = fit_model(train_part)
    predictions = predict_model(fold_model, val_part)
    score = mape(val_part["num_sold"].to_numpy(), predictions)

    print("Prediction min:", predictions.min())
    print("Prediction max:", predictions.max())
    print("Prediction mean:", predictions.mean())
    print("Negative predictions:", int((predictions < 0).sum()))
    print("All predictions finite:", bool(np.isfinite(predictions).all()))
    print(f"MAPE: {score:.6f}%")

    fold_scores.append({
        "fold": fold_name,
        "mape": score,
    })

fold_metrics = pd.DataFrame(fold_scores)
mean_expanding_mape = float(fold_metrics["mape"].mean())

print("\nExpanding-window results:")
print(fold_metrics.to_string(index=False))
print(f"Mean expanding-window MAPE: {mean_expanding_mape:.6f}%")


Training fold_2_2015


Clipping 456 negative predictions to zero (raw minimum: -10.996212).
Prediction min: 0.0
Prediction max: 4352.1163108688415
Prediction mean: 778.3799056108796
Negative predictions: 0
All predictions finite: True
MAPE: 23.756415%

Training fold_3_2016


Clipping 574 negative predictions to zero (raw minimum: -16.554488).
Prediction min: 0.0
Prediction max: 3631.4265722886576
Prediction mean: 691.1233500168712
Negative predictions: 0
All predictions finite: True
MAPE: 15.059582%

Expanding-window results:
       fold      mape
fold_1_2014 15.917685
fold_2_2015 23.756415
fold_3_2016 15.059582
Mean expanding-window MAPE: 18.244561%


## 5. Long-Horizon Evaluation

In [7]:
print("Training long_horizon")

long_model = fit_model(long_train)
long_predictions = predict_model(long_model, long_val)

long_horizon_mape = mape(
    long_val["num_sold"].to_numpy(),
    long_predictions,
)

print("Prediction min:", long_predictions.min())
print("Prediction max:", long_predictions.max())
print("Prediction mean:", long_predictions.mean())
print("Negative predictions:", int((long_predictions < 0).sum()))
print("All predictions finite:", bool(np.isfinite(long_predictions).all()))
print(f"Long-horizon MAPE: {long_horizon_mape:.6f}%")

Training long_horizon


Clipping 2239 negative predictions to zero (raw minimum: -19.865019).
Prediction min: 0.0
Prediction max: 4925.9072151098235
Prediction mean: 829.7589022372127
Negative predictions: 0
All predictions finite: True
Long-horizon MAPE: 22.522954%


## 6. Baseline Comparison

In [8]:
metrics_path = PROJECT_ROOT / "results" / "metrics.csv"
saved_metrics = pd.read_csv(metrics_path)

baseline_names = [
    "group_median",
    "seasonal_naive_last_year",
    "ridge_calendar_one_hot",
]

baseline_metrics = saved_metrics.loc[
    saved_metrics["model"].isin(baseline_names)
    & saved_metrics["version"].eq("v1")
].copy()

expected_folds = {
    "expanding_window": {
        "fold_1_2014",
        "fold_2_2015",
        "fold_3_2016",
    },
    "long_horizon": {"2014_2016"},
}

for model_name in baseline_names:
    for scheme, expected in expected_folds.items():
        rows = baseline_metrics.loc[
            baseline_metrics["model"].eq(model_name)
            & baseline_metrics["validation_scheme"].eq(scheme)
        ]
        assert len(rows) == len(expected), (model_name, scheme)
        assert set(rows["fold"]) == expected, (model_name, scheme)

comparison = (
    baseline_metrics
    .groupby(["model", "validation_scheme"])["mape"]
    .mean()
    .unstack("validation_scheme")
)

comparison.loc[MODEL_NAME, "expanding_window"] = mean_expanding_mape
comparison.loc[MODEL_NAME, "long_horizon"] = long_horizon_mape

print(comparison.round(6).to_string())

best_baseline = "seasonal_naive_last_year"
difference = comparison.loc[MODEL_NAME] - comparison.loc[best_baseline]

print("\nMAPE difference versus seasonal naive (percentage points):")
print(difference.round(6).to_string())

validation_scheme         expanding_window  long_horizon
model                                                   
group_median                     16.319486     17.460368
ridge_calendar_one_hot           17.859827     23.907846
seasonal_naive_last_year         13.273855     17.019122
catboost_raw                     18.244561     22.522954

MAPE difference versus seasonal naive (percentage points):
validation_scheme
expanding_window    4.970706
long_horizon        5.503832


## 7. Save Results

In [9]:
import json

experiment_params = {
    **model_2014.get_params(),
    "features": FEATURE_COLS,
    "target_transform": "none",
    "prediction_postprocessing": "clip_min_0",
    "catboost_version": catboost.__version__,
}

params_json = json.dumps(experiment_params, sort_keys=True)

evaluation_results = [
    (
        "expanding_window",
        row["fold"],
        train_part,
        val_part,
        float(row["mape"]),
    )
    for row, (train_part, val_part) in zip(fold_scores, folds, strict=True)
]

evaluation_results.append(
    (
        "long_horizon",
        "2014_2016",
        long_train,
        long_val,
        float(long_horizon_mape),
    )
)

records = []

for scheme, fold, train_part, val_part, score in evaluation_results:
    records.append({
        "model": MODEL_NAME,
        "version": MODEL_VERSION,
        "validation_scheme": scheme,
        "fold": fold,
        "train_period": (
            f"{train_part['date'].min().date()} to "
            f"{train_part['date'].max().date()}"
        ),
        "validation_period": (
            f"{val_part['date'].min().date()} to "
            f"{val_part['date'].max().date()}"
        ),
        "mape": round(score, 6),
        "params": params_json,
    })

proposed_metrics = pd.DataFrame(records)

assert list(proposed_metrics.columns) == list(saved_metrics.columns)
assert len(proposed_metrics) == 4
assert np.isfinite(proposed_metrics["mape"].to_numpy()).all()
assert not proposed_metrics.duplicated(
    ["model", "version", "validation_scheme", "fold"]
).any()

print(proposed_metrics.drop(columns="params").to_string(index=False))
print("\nParameters:")
print(json.dumps(experiment_params, indent=2, sort_keys=True))
print("\nMetrics prepared. No files written.")

       model version validation_scheme        fold             train_period        validation_period      mape
catboost_raw      v1  expanding_window fold_1_2014 2010-01-01 to 2013-12-31 2014-01-01 to 2014-12-31 15.917685
catboost_raw      v1  expanding_window fold_2_2015 2010-01-01 to 2014-12-31 2015-01-01 to 2015-12-31 23.756415
catboost_raw      v1  expanding_window fold_3_2016 2010-01-01 to 2015-12-31 2016-01-01 to 2016-12-31 15.059582
catboost_raw      v1      long_horizon   2014_2016 2010-01-01 to 2013-12-31 2014-01-01 to 2016-12-31 22.522954

Parameters:
{
  "allow_writing_files": false,
  "cat_features": [
    "country",
    "store",
    "product"
  ],
  "catboost_version": "1.2.10",
  "depth": 6,
  "features": [
    "year",
    "month",
    "day",
    "day_of_week",
    "day_of_year",
    "week_of_year",
    "quarter",
    "is_weekend",
    "time_idx",
    "day_of_week_sin",
    "day_of_week_cos",
    "month_sin",
    "month_cos",
    "day_of_year_sin",
    "day_of_year_cos",


In [10]:
current_metrics = pd.read_csv(metrics_path)
original_bytes = metrics_path.read_bytes()

assert list(current_metrics.columns) == list(proposed_metrics.columns)

own_rows = (
    current_metrics["model"].eq(MODEL_NAME)
    & current_metrics["version"].eq(MODEL_VERSION)
)

if own_rows.any():
    sort_columns = ["validation_scheme", "fold"]

    existing = (
        current_metrics.loc[own_rows]
        .sort_values(sort_columns)
        .reset_index(drop=True)
    )
    expected = (
        proposed_metrics
        .sort_values(sort_columns)
        .reset_index(drop=True)
    )

    pd.testing.assert_frame_equal(
        existing,
        expected,
        check_dtype=False,
        check_exact=True,
    )
    print("Identical results already exist. No files changed.")
else:
    with metrics_path.open("a", encoding="utf-8", newline="") as output:
        if original_bytes and not original_bytes.endswith((b"\n", b"\r")):
            output.write("\n")

        proposed_metrics.to_csv(
            output,
            index=False,
            header=False,
            lineterminator="\n",
        )

    assert metrics_path.read_bytes().startswith(original_bytes)
    print("Added 4 proposed-model rows. Existing rows preserved.")

updated_metrics = pd.read_csv(metrics_path)
print("Total metric rows:", len(updated_metrics))

Identical results already exist. No files changed.
Total metric rows: 16


## 8. Conclusions

CatBoost v1 achieved 18.24% mean expanding-window MAPE and 22.52%
long-horizon MAPE. The seasonal-naive baseline performed better:
13.27% and 17.02%, respectively.

This version did not improve on the strongest baseline. Negative predictions
were clipped to zero in all splits. Training used RMSE on the original target,
while evaluation used MAPE.

The model provides a starting point for the improvement stage.
Possible next steps include a log-target transformation, parameter tuning,
and additional features that do not use future sales.